Imports

In [3]:
# ── CELL 1: Imports ──────────────────────────────────────────────────────────
# joblib  → to load the saved RF model (.pkl file)
# pandas  → to structure input data for prediction
# numpy   → numerical operations

import joblib
import pandas as pd
import numpy as np

print("✅ Libraries imported")

✅ Libraries imported


 Upload & Load Model

In [4]:
# ── CELL 2: Load Model from Google Drive ────────────────────────────────────
# For large model files, Google Drive is more reliable than direct upload.
# Mount Drive → load directly from the path.

from google.colab import drive
import joblib

# Mount your Google Drive
drive.mount("/content/drive")

# Load the model directly from Drive
# Update the path if you saved it in a subfolder
model_path = "/content/drive/MyDrive/pm25_rf_model.pkl"

rf_model = joblib.load(model_path)

print("✅ Model loaded successfully")
print(f"   Model type   : {type(rf_model)}")
print(f"   No. of trees : {rf_model.n_estimators}")
print(f"   No. features : {rf_model.n_features_in_}")

KeyboardInterrupt: 

Define Features

In [ ]:
# ── CELL 3: Define Feature Names ─────────────────────────────────────────────
# Must match exactly what was used during training — same names, same order.

features = ["PM10", "SO2", "NO2", "CO", "O3",
            "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

print("✅ Features ready:", features)

 Predict Function

In [ ]:
# ── CELL 4: Prediction + AQI Status Function ─────────────────────────────────
# Reusable function — pass any input, get PM2.5 prediction + AQI label.
# AQI thresholds based on WHO / CPCB guidelines.

def predict_pm25(input_dict):
    input_df = pd.DataFrame([input_dict])[features]
    pred     = rf_model.predict(input_df)[0]

    if pred <= 15:
        status = "🟢 Good"
    elif pred <= 35:
        status = "🟡 Moderate"
    elif pred <= 75:
        status = "🟠 Unhealthy for Sensitive Groups"
    else:
        status = "🔴 Unhealthy / Hazardous"

    return pred, status

Single Prediction

In [ ]:
# ── CELL 5: Single Prediction ────────────────────────────────────────────────
# One set of readings → one PM2.5 prediction.
# Change values below to predict for any conditions.

reading = {
    "PM10": 120.0,
    "SO2" : 10.0,
    "NO2" : 40.0,
    "CO"  : 800.0,
    "O3"  : 60.0,
    "TEMP": 15.0,
    "PRES": 1010.0,
    "DEWP": 5.0,
    "RAIN": 0.0,
    "WSPM": 2.0
}

pred, status = predict_pm25(reading)

print("📥 Input:")
for k, v in reading.items():
    print(f"   {k:<6} : {v}")

print(f"\n📤 Predicted PM2.5 : {pred:.2f} μg/m³")
print(f"   AQI Status     : {status}")

Multiple Predictions

In [ ]:
# ── CELL 6: Multiple Predictions ─────────────────────────────────────────────
# 3 scenarios tested at once:
#   Sample 1 → moderate pollution (typical city day)
#   Sample 2 → high pollution    (winter, low wind)
#   Sample 3 → clean air         (rain, high wind)

from tabulate import tabulate

samples = pd.DataFrame({
    "PM10": [120.0,  250.0,  50.0],
    "SO2" : [ 10.0,   35.0,   5.0],
    "NO2" : [ 40.0,   80.0,  20.0],
    "CO"  : [800.0, 1500.0, 400.0],
    "O3"  : [ 60.0,   30.0,  90.0],
    "TEMP": [ 15.0,    5.0,  25.0],
    "PRES": [1010.0, 1015.0, 1005.0],
    "DEWP": [  5.0,   -2.0,  15.0],
    "RAIN": [  0.0,    0.0,   0.2],
    "WSPM": [  2.0,    0.5,   4.0],
})

preds = rf_model.predict(samples[features])

results = pd.DataFrame({
    "Scenario"        : ["Moderate", "High Pollution", "Clean Air"],
    "PM10"            : samples["PM10"].values,
    "CO"              : samples["CO"].values,
    "WSPM"            : samples["WSPM"].values,
    "Predicted PM2.5" : preds.round(2),
    "AQI Status"      : [
        "🟢 Good" if p <= 15
        else "🟡 Moderate" if p <= 35
        else "🟠 Unhealthy-Sensitive" if p <= 75
        else "🔴 Unhealthy"
        for p in preds
    ]
})

print("📊 Prediction Results:\n")
print(tabulate(results, headers="keys", tablefmt="rounded_outline",
               showindex=False, numalign="center"))

 Live User Input

In [ ]:
# ── CELL 7: Live User Input Prediction ───────────────────────────────────────
# Type in real values when prompted.
# Simulates how this model would work in a real deployment.

print("🌫️  PM2.5 Predictor — Enter Readings\n")

try:
    reading = {
        "PM10": float(input("PM10  (μg/m³) : ")),
        "SO2" : float(input("SO2   (μg/m³) : ")),
        "NO2" : float(input("NO2   (μg/m³) : ")),
        "CO"  : float(input("CO    (μg/m³) : ")),
        "O3"  : float(input("O3    (μg/m³) : ")),
        "TEMP": float(input("TEMP  (°C)    : ")),
        "PRES": float(input("PRES  (hPa)   : ")),
        "DEWP": float(input("DEWP  (°C)    : ")),
        "RAIN": float(input("RAIN  (mm)    : ")),
        "WSPM": float(input("WSPM  (m/s)   : ")),
    }

    pred, status = predict_pm25(reading)

    print(f"\n{'═'*40}")
    print(f"  📤 Predicted PM2.5 : {pred:.2f} μg/m³")
    print(f"  AQI Status         : {status}")
    print(f"{'═'*40}")

except ValueError:
    print("⚠️  Please enter numbers only")